# Sepformer Audio Separation
This notebook runs the SpeechBrain Sepformer model on pre-normalized audio files (`16kHz`, `mono`, `.wav`), avoiding the need to install `ffmpeg` or do any raw file processing remotely.

Before running, make sure to:
1. Switch your Colab runtime to **GPU** (Runtime -> Change runtime type -> Hardware accelerator -> GPU).
2. Upload your pre-normalized audio files to `data/normalized` or mount your Google Drive having those files.


In [1]:
%pip install "speechbrain==1.0.3" "torchaudio==2.8.*" "torch==2.8.*" "tqdm>=4.67.3" "huggingface_hub<1.0.0"

In [2]:
from pathlib import Path
import torchaudio
import torch
import gc
from speechbrain.inference.separation import SepformerSeparation as separator

# If running in Colab/Jupyter, tqdm.notebook looks nicer
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# Directories
NORM_DIR = Path("data/normalized")
OUT_DIR = Path("out_separated")
MODEL_SOURCE = "speechbrain/sepformer-wsj02mix"

# Ensure directories exist
NORM_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

/usr/local/lib/python3.12/dist-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _speechbrain_save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _speechbrain_load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoi

In [3]:
print(f"Loading Sepformer model from {MODEL_SOURCE}...")

# Use GPU if available
run_opts = {"device": "cuda"} if torch.cuda.is_available() else {"device": "cpu"}
model = separator.from_hparams(source=MODEL_SOURCE, run_opts=run_opts)

print("Model loaded successfully.")

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/sepformer-wsj02mix' if not cached


Loading Sepformer model from speechbrain/sepformer-wsj02mix...


hyperparams.yaml: 0.00B [00:00, ?B/s]

INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/sepformer-wsj02mix' if not cached
/usr/local/lib/python3.12/dist-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
DEBUG:speechbrain.utils.parameter_transfer:Fetching files for pretraining (no collection directory set)
INFO:speechbrain.utils.fetching:Fetch masknet.ckpt: Fetching from HuggingFace Hub 'speechbrain/sepformer-wsj02mix' if not cached


masknet.ckpt:   0%|          | 0.00/113M [00:00<?, ?B/s]

...

encoder.ckpt:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["encoder"] = /root/.cache/huggingface/hub/models--speechbrain--sepformer-wsj02mix/snapshots/3a2826343a10e2d2e8a75f79aeab5ff3a2473531/encoder.ckpt
INFO:speechbrain.utils.fetching:Fetch decoder.ckpt: Fetching from HuggingFace Hub 'speechbrain/sepformer-wsj02mix' if not cached


decoder.ckpt:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["decoder"] = /root/.cache/huggingface/hub/models--speechbrain--sepformer-wsj02mix/snapshots/3a2826343a10e2d2e8a75f79aeab5ff3a2473531/decoder.ckpt
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: masknet, encoder, decoder
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): masknet -> /root/.cache/huggingface/hub/models--speechbrain--sepformer-wsj02mix/snapshots/3a2826343a10e2d2e8a75f79aeab5ff3a2473531/masknet.ckpt
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): encoder -> /root/.cache/huggingface/hub/models--speechbrain--sepformer-wsj02mix/snapshots/3a2826343a10e2d2e8a75f79aeab5ff3a2473531/encoder.ckpt
DEBUG:speechbrain.utils.parameter_transfer:Redirecting (loading from local path): decoder -> /root/.cache/huggingface/hub/models--speechbrain--sepformer-wsj02mix/snapshots/3a2826343a10e2d2e8a75f79aeab5ff3a2473531/decoder.ckpt


Model loaded successfully.


In [4]:
# Find all pre-normalized .wav files in NORM_DIR
norm_path = Path("/content/normalized")

audio_files = sorted([p for p in norm_path.glob("*.wav") if p.is_file()])

if not audio_files:
    print(f"No .wav files found in {norm_path}")
else:
    print(f"Found {len(audio_files)} files to process.")
    for p in audio_files:
        print(p)

Found 25 files to process.
/content/normalized/conv__+33327866663_20-08-2024_8_58_10.wav
/content/normalized/conv__+4915203230182_22-08-2024_8_06_41.wav
/content/normalized/conv__+4916095913442_20-08-2024_11_45_32.wav
/content/normalized/conv__+491729920245_24-07-2024_10_15_48.wav
/content/normalized/conv__+491732965552_22-08-2024_9_46_35.wav
/content/normalized/conv__+491779108125_26-08-2024_10_23_16.wav
/content/normalized/conv__+49620363603_28-08-2024_9_28_15.wav
/content/normalized/conv__+49626137904_10-07-2024_8_38_40.wav
/content/normalized/conv__+49661922317_05-08-2024_8_39_24.wav
/content/normalized/conv__+49706221952_07-08-2024_14_25_06.wav
/content/normalized/conv__+49706262738_21-08-2024_10_58_04.wav
/content/normalized/conv__+49706353138_26-07-2024_9_13_32.wav
/content/normalized/conv__+49711621826_06-08-2024_9_06_00.wav
/content/normalized/conv__+49711892461_06-08-2024_8_48_05.wav
/content/normalized/conv__+49713146656_20-08-2024_9_13_44.wav
/content/normalized/conv__+4971

In [7]:
from tqdm import tqdm

def separate_long_audio(model, audio_tensor, sample_rate, chunk_len=30.0, overlap=15.0):
    """
    Separates long audio using overlapping chunks.
    Keeps the heavy 'out_tensor' in CPU RAM to prevent GPU OOM,
    while leveraging GPU for the individual forward passes on 'chunk'.
    """
    chunk_samples = int(chunk_len * sample_rate)
    overlap_samples = int(overlap * sample_rate)
    step = chunk_samples - overlap_samples
    
    T = audio_tensor.shape[1]
    
    # If the audio is shorter than a chunk, just separate it right away
    if T <= chunk_samples:
        with torch.no_grad():
            est = model.separate_batch(audio_tensor).detach().cpu()
            est = est / est.abs().max(dim=1, keepdim=True)[0]
            return est
            
    # VERY IMPORTANT: Keep the big output tensor on 'cpu' RAM so you don't OOM the colab GPU
    out_tensor = torch.zeros(2, T, device='cpu')
    fade_in = torch.linspace(0, 1, overlap_samples, device='cpu').unsqueeze(-1)
    fade_out = torch.linspace(1, 0, overlap_samples, device='cpu').unsqueeze(-1)
    
    prev_raw = None
    
    for i in range(0, T, step):
        end = min(i + chunk_samples, T)
        chunk = audio_tensor[:, i:end]
        
        # separate_batch will automatically use the GPU because 'model' is on the GPU.
        # We immediately .detach().cpu() to move the small chunk result back to main RAM.
        with torch.no_grad():
            est_sources = model.separate_batch(chunk).detach().cpu()[0] 
        
        curr_time = est_sources.shape[0]
        
        if prev_raw is not None:
            end_prev = i - step + prev_raw.shape[0]
            ov_size = min(end_prev - i, curr_time)
            
            if ov_size > 0:
                prev_overlap = prev_raw[-ov_size:]
                curr_overlap = est_sources[:ov_size]
                
                diff_0 = torch.abs(prev_overlap - curr_overlap).mean()
                diff_1 = torch.abs(prev_overlap - curr_overlap.flip(dims=[1])).mean()
                
                if diff_1 < diff_0:
                    est_sources = est_sources.flip(dims=[1])
                    
        prev_raw = est_sources.clone()
        weight = torch.ones(curr_time, 1, device='cpu')
        
        if i > 0:
            ov_size = min(overlap_samples, curr_time)
            weight[:ov_size] *= fade_in[:ov_size]
            
        if end < T:
            ov_size = min(overlap_samples, curr_time)
            weight[-ov_size:] *= fade_out[-ov_size:]
            
        est_sources *= weight
        out_tensor[:, i:end] += est_sources.transpose(0, 1)
        
        del chunk
        del est_sources
        del weight
        gc.collect()
        
        # Free GPU memory after each chunk
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    out_tensor = out_tensor.transpose(0, 1).unsqueeze(0)
    out_tensor = out_tensor / out_tensor.abs().max(dim=1, keepdim=True)[0]
    
    return out_tensor

# Update loop to use the chunking function
pbar = tqdm(audio_files, desc="Separating audio")
for norm_path in pbar:
    recording_id = norm_path.stem
    pbar.set_postfix(file=recording_id)
    
    rec_out_dir = OUT_DIR / recording_id
    rec_out_dir.mkdir(parents=True, exist_ok=True)
    
    out_spk1 = rec_out_dir / f"{recording_id}_spk1.wav"
    out_spk2 = rec_out_dir / f"{recording_id}_spk2.wav"
    
    # Skip if already processed
    if out_spk1.exists() and out_spk2.exists():
        continue
    try:
        sample_rate = 16000
        
        # Load the normalized file into a tensor
        batch, fs_file = torchaudio.load(str(norm_path))
        if fs_file != sample_rate:
            tf = torchaudio.transforms.Resample(orig_freq=fs_file, new_freq=sample_rate)
            batch = tf(batch.mean(dim=0, keepdim=True))
        
        # Clean RAM / VRAM before we begin parsing a very long audio file
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        # >> MAGIC HAPPENS HERE: Separates overlapping chunks using the GPU
        est_sources = separate_long_audio(model, batch, sample_rate, chunk_len=20.0, overlap=10.0)
        
        # Save separated tracks
        torchaudio.save(str(out_spk1), est_sources[:, :, 0], sample_rate)
        torchaudio.save(str(out_spk2), est_sources[:, :, 1], sample_rate)
        
        del batch
        del est_sources
        gc.collect()
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    except Exception as e:
        print(f"\nError separating full file {recording_id}: {e}")
    

Separating audio:   0%|          | 0/25 [00:00<?, ?it/s, file=conv__+33327866663_20-08-2024_8_58_10]/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into To

In [8]:
!zip -r files.zip /content/out_separated

  adding: content/out_separated/ (stored 0%)
  adding: content/out_separated/conv__+49711892461_06-08-2024_8_48_05/ (stored 0%)
  adding: content/out_separated/conv__+49711892461_06-08-2024_8_48_05/conv__+49711892461_06-08-2024_8_48_05_spk2.wav (deflated 21%)
  adding: content/out_separated/conv__+49711892461_06-08-2024_8_48_05/conv__+49711892461_06-08-2024_8_48_05_spk1.wav (deflated 25%)
  adding: content/out_separated/conv__+491732965552_22-08-2024_9_46_35/ (stored 0%)
  adding: content/out_separated/conv__+491732965552_22-08-2024_9_46_35/conv__+491732965552_22-08-2024_9_46_35_spk2.wav (deflated 29%)
  adding: content/out_separated/conv__+491732965552_22-08-2024_9_46_35/conv__+491732965552_22-08-2024_9_46_35_spk1.wav (deflated 32%)
  adding: content/out_separated/conv__+49713215540_05-07-2024_10_34_47/ (stored 0%)
  adding: content/out_separated/conv__+49713215540_05-07-2024_10_34_47/conv__+49713215540_05-07-2024_10_34_47_spk2.wav (deflated 21%)
  adding: content/out_separated/conv__